# 04. 미래 주가 예측 (Recursive Extension)

## 📋 개요
학습된 Multi-horizon 모델을 사용하여 미래 주가를 예측합니다.

## ✨ 핵심 전략: Recursive Extension
- **Step 1**: 최신 데이터로 5일치(h1~h5) 예측
- **Step 2**: 예측값을 실제값처럼 사용하여 다음 5일 예측
- **Step 3**: 목표 기간까지 반복 확장

```
현재 데이터 → [모델] → 5일 예측값
예측값 + 현재 데이터 → [모델] → 다음 5일 예측값
예측값 + ... → [모델] → 계속 확장...
```

## 🔧 패치 이력
- **v1.1** (2026-02-08): Chunk 데이터 오염 방지 — volume 최근 평균 사용
- **v1.2** (현재): v3.6.0 피처 스키마 동기화 + 매크로/정적 피처 미래값 반영
  - `calculate_features_for_ticker` → builder.py v3.6.0 스키마와 일치
  - 매크로 피처(`feature_kospi` 등) 97단계 추정값 로드 및 new_row 반영
  - 정적 피처(`feature_is_kospi`) 및 캘린더 피처(`feature_is_monday/friday`) new_row 반영

## 🔧 Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings

from src.utils.config import load_config, ProjectPaths, is_ensemble, resolve_model_name
from src.features.technical import (
    calc_sma, calc_rsi, calc_macd, calc_bollinger, calc_volume_ratio
)

warnings.filterwarnings('ignore')

## 1️⃣ 설정 및 경로

In [ ]:
cfg   = load_config()
paths = ProjectPaths.from_config(cfg)
paths.ensure_dirs()

meta_dir = Path(cfg['paths'].get('meta_dir', 'data/99_meta'))

# 매크로 관련 경로
macro_hist_path     = meta_dir / 'macro_regime.parquet'
macro_forecast_path = meta_dir / 'macro_regime_forecast.parquet'
calendar_path       = meta_dir / 'krx_calendar.csv'

## 2️⃣ 파일 존재 확인

In [ ]:
model_path   = paths.get_model_path()
dataset_path = paths.get_dataset_parquet()

print(f"✅ 모델 파일:    {model_path.name}")
print(f"✅ 데이터셋:     {dataset_path}")
print(f"✅ 매크로 실측:  {macro_hist_path}  존재={macro_hist_path.exists()}")
print(f"✅ 매크로 추정:  {macro_forecast_path}  존재={macro_forecast_path.exists()}")

if not macro_forecast_path.exists():
    print("⚠️  macro_regime_forecast.parquet 없음 — 97단계(97_forecast_macro.ipynb)를 먼저 실행하세요.")
    print("   매크로 피처가 NaN으로 채워져 예측 품질이 저하됩니다.")

## 3️⃣ 매크로 데이터 로드 (실측 + 추정 통합)

### ✨ v1.2 신설
과거 실측(`macro_regime.parquet`)과 97단계 미래 추정값(`macro_regime_forecast.parquet`)을
합쳐 date → macro 피처 조회 테이블을 구성합니다.

In [ ]:
# ── 과거 실측 로드 ────────────────────────────────────────────────────────
df_macro = pd.read_parquet(macro_hist_path)
df_macro['date'] = pd.to_datetime(df_macro['date'])

# ── 미래 추정값 병합 (존재할 때만) ───────────────────────────────────────
if macro_forecast_path.exists():
    df_macro_fc = pd.read_parquet(macro_forecast_path)
    df_macro_fc['date'] = pd.to_datetime(df_macro_fc['date'])
    df_macro = pd.concat([df_macro, df_macro_fc], ignore_index=True)
    df_macro = df_macro.drop_duplicates('date')  # 실측이 우선(먼저 concat되었으므로)
    print(f"✅ 매크로 데이터 통합 완료: {len(df_macro)}일 ({df_macro['date'].min().date()} ~ {df_macro['date'].max().date()})")
else:
    print(f"⚠️  추정값 없이 실측만 사용: {len(df_macro)}일")

# ── date → macro 피처 조회 테이블 구성 ────────────────────────────────────
# 02_build_dataset.ipynb 과 동일하게 feature_ 접두어를 붙여 관리
MACRO_COLS = ['kospi', 'usd_krw', 'vix', 'us_return_1d', 'market_regime']

# 실제 parquet에 존재하는 컬럼만 사용 (컬럼 구성이 달라질 경우 대비)
macro_available_cols = [c for c in MACRO_COLS if c in df_macro.columns]

df_macro_lookup = df_macro[['date'] + macro_available_cols].set_index('date')

# 피처명 매핑: 'kospi' → 'feature_kospi'  (02단계와 동일한 변환)
MACRO_FEATURE_MAP = {col: f'feature_{col}' for col in macro_available_cols}

print(f"\n📌 매크로 피처 매핑:")
for k, v in MACRO_FEATURE_MAP.items():
    print(f"   {k:20s} → {v}")

## 4️⃣ 데이터 로드

In [ ]:
# ── Feature 데이터셋 로드 ─────────────────────────────────────────────────
print("📥 Feature 데이터 로드 중...")
df_features = pd.read_parquet(dataset_path)
df_features['date'] = pd.to_datetime(df_features['date'])
df_features = df_features.sort_values(['ticker', 'date']).reset_index(drop=True)

last_date = df_features['date'].max()
print(f"   - 총 행수: {len(df_features):,}")
print(f"   - 종목 수: {df_features['ticker'].nunique()}")
print(f"   - 최신 날짜: {last_date.strftime('%Y-%m-%d')}")

# ── ticker → is_kospi 정적 정보 추출 ─────────────────────────────────────
# df_features에 이미 feature_is_kospi 컬럼이 존재
IS_KOSPI_COL = 'feature_is_kospi'
if IS_KOSPI_COL in df_features.columns:
    ticker_is_kospi = (
        df_features.groupby('ticker')[IS_KOSPI_COL].last().to_dict()
    )
else:
    print(f"⚠️  {IS_KOSPI_COL} 컬럼 없음 — 0으로 대체")
    ticker_is_kospi = {}

# ── 영업일 캘린더 로드 ────────────────────────────────────────────────────
print("\n📅 영업일 캘린더 로드 중...")
df_calendar = pd.read_csv(calendar_path)
df_calendar['date'] = pd.to_datetime(df_calendar['date'])

future_dates = (
    df_calendar[df_calendar['date'] > last_date]['date']
    .sort_values().reset_index(drop=True)
)

if len(future_dates) == 0:
    raise ValueError(
        f"❌ 예측할 미래 영업일이 없습니다.\n"
        f"   최신 데이터: {last_date.strftime('%Y-%m-%d')}\n"
        f"   캘린더 종료일: {df_calendar['date'].max().strftime('%Y-%m-%d')}\n"
        "💡 Tip: 99_save_trading_days.ipynb에서 기간을 연장하세요."
    )

print(f"   - 총 예측일수: {len(future_dates)}일")

## 5️⃣ 모델 로드 (`active_model` 기반 클래스메서드 사용)

In [ ]:
# ============================================================
# 04_forecast_future.ipynb — load_model 셀 패치 (v3.8.0)
#
# 변경 내역:
#   - active_model 문자열 하드코딩 비교 제거
#   - is_ensemble() / resolve_model_name() 기반 동적 로드로 교체
#   - 약칭('lgbm', 'rf') 및 앙상블 조합('mlp+rf' 등) 모두 처리
# ============================================================

active_model = cfg.get('active_model', 'lightgbm')

if is_ensemble(active_model):
    # 앙상블: 조합 문자열에 관계없이 EnsembleModel로 로드
    from src.models.ensemble_model import EnsembleModel
    model = EnsembleModel.load(str(model_path))

else:
    # 단일 모델: 정칭/약칭 모두 canonical로 정규화 후 분기
    canonical, _ = resolve_model_name(active_model)

    if canonical == 'lightgbm':
        from src.models.lightgbm_model import LightGBMModel
        model = LightGBMModel.load(str(model_path))
    elif canonical == 'randomforest':
        from src.models.randomforest_model import RandomForestMultiModel
        model = RandomForestMultiModel.load(str(model_path))
    elif canonical == 'mlp':
        from src.models.mlp_model import MLPModel
        model = MLPModel.load(str(model_path))

target_type         = cfg['training'].get('target_type', 'log_close')
actual_target_names = [f'target_log_close_h{h}' for h in cfg['training']['horizons']]

CHUNK_SIZE = len(cfg['training']['horizons'])
NUM_CHUNKS = int(np.ceil(len(future_dates) / CHUNK_SIZE))
horizons     = cfg['training']['horizons']
feature_cols = model.feature_list   # 모델 학습 시 사용한 피처 목록

print(f"✅ 모델 로드 완료 ({active_model})")
print(f"   - target_type: {target_type}")
print(f"   - horizons: {horizons}")
print(f"   - CHUNK_SIZE: {CHUNK_SIZE}, NUM_CHUNKS: {NUM_CHUNKS}")
print(f"   - feature 수: {len(feature_cols)}")

## 6️⃣ Feature 생성 함수 정의

### ✨ v1.2 변경 (v3.6.0 스키마 동기화)

| 이전 | 이후 | 사유 |
|------|------|------|
| `feature_ma_{w}` (절대가격) | `feature_ma_{w}_disparity` (이격도) | builder.py 일치 |
| `feature_bb_upper/lower` (절대가격) | `feature_bb_pct_b`, `feature_bb_width` | builder.py 일치 |
| `liquidity_score` (원화) | `feature_log_liquidity` | builder.py 일치 |

> **매크로/정적/캘린더 피처는 이 함수 밖에서 처리합니다.**  
> 이 함수는 `close`, `volume` 기반 기술적 지표만 담당합니다.

In [ ]:
def calculate_features_for_ticker(df_ticker: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    단일 종목의 기술적 지표 재계산 — builder.py v3.6.0 스키마와 동일.

    매크로/정적/캘린더 피처는 건드리지 않음.
    (해당 컬럼은 과거 행에서 유지되고, 미래 행은 new_row 생성 시 주입됨)
    """
    df = df_ticker.copy()
    params = config['preprocessing']

    # 1. 이동평균 이격도 (절대가격 → 무차원)
    for window in params['technical_windows']:  # [5, 60]
        ma = calc_sma(df['close'], window)
        df[f'feature_ma_{window}_disparity'] = (df['close'] / ma) - 1.0

    # 2. 변동성 (20일)
    df['feature_volatility_20'] = df['close'].pct_change().rolling(20).std()

    # 3. 거래량 비율
    df['feature_volume_ratio'] = calc_volume_ratio(df['volume'], params['volume_window'])

    # 4. RSI
    df['feature_rsi_14'] = calc_rsi(df['close'], params['rsi_period'])

    # 5. MACD
    macd, signal, hist = calc_macd(df['close'])
    df['feature_macd']        = macd
    df['feature_macd_signal'] = signal
    df['feature_macd_hist']   = hist

    # 6. 볼린저 밴드 → %B, Width (절대가격 → 무차원)
    upper, mid, lower = calc_bollinger(df['close'])
    df['feature_bb_pct_b'] = (df['close'] - lower) / (upper - lower + 1e-9)
    df['feature_bb_width']  = (upper - lower) / (mid + 1e-9)

    # 7. 유동성 (절대 거래대금 → 로그 변환)
    liquidity = (df['close'] * df['volume']).rolling(20).mean()
    df['feature_log_liquidity'] = np.log1p(liquidity)

    # 8. 운영용 메타 (feature_ 접두어 없음 — 모델 피처로 사용되지 않음)
    df['liquidity_score']  = liquidity
    df['risk_composite']   = df['feature_volatility_20'].fillna(0)

    return df


def get_macro_row(pred_date: pd.Timestamp) -> dict:
    """
    주어진 날짜의 매크로 피처값을 반환.
    df_macro_lookup (date-indexed)를 참조하며, 없으면 직전 날짜 값 사용(ffill).
    """
    if pred_date in df_macro_lookup.index:
        row = df_macro_lookup.loc[pred_date]
    else:
        # 해당 날짜 이전 마지막 값
        past = df_macro_lookup[df_macro_lookup.index <= pred_date]
        if past.empty:
            return {v: np.nan for v in MACRO_FEATURE_MAP.values()}
        row = past.iloc[-1]
    return {MACRO_FEATURE_MAP[col]: row[col] for col in macro_available_cols}


print("✅ Feature 계산 함수 정의 완료")

## 7️⃣ Recursive Extension 예측 실행

### ✨ v1.2 변경: new_row 피처 완전 채우기

| 피처 그룹 | 처리 방식 |
|-----------|----------|
| 기술적 지표 | `calculate_features_for_ticker` 재계산 (close 기반) |
| 매크로 피처 | `get_macro_row(pred_date)` 조회 후 new_row 주입 |
| `feature_is_kospi` | ticker별 정적 값 유지 |
| `feature_is_monday/friday` | `pred_date.weekday()` 계산 |

In [ ]:
# ============================================================
# 04_forecast_future.ipynb — recursive_predict 셀 패치 (v3.8.0)
#
# 변경 내역:
#   - 종목별 루프를 try/except로 감싸 예측 실패 시 해당 종목 건너뜀
#   - 실패 종목 목록(ticker, 에러 메시지)을 skipped_tickers에 수집
#   - 루프 완료 후 제외 종목 수 및 목록 출력
# ============================================================

# ==========================================
# 04단계 Recursive Extension — log_return_1d 분기 (v3.9.1)
# ==========================================
#
# [log_close 모드] (기존, 유지)
#   모델 출력: log(close(t+n)) → exp()로 바로 역산
#   chunk 간 기준가: 필요 없음 (절대값 직접 예측)
#
# [log_return_1d 모드] (v3.9.1 사다리꼴 보정)
#   수정 전:  y(t+h) = y(t) + cumsum_h
#   수정 후:  y(t+h) = y(t) + cumsum_h + (Δy(t) − Δy(t+h)) / 2
#
#   Δy(t)    = prev_delta_y  (앵커 — 종목 루프 외부 초기화, 각 chunk 후 갱신)
#              - 첫 chunk: df_ticker 마지막 행의 실측 log return
#              - 이후 chunk: 직전 chunk 마지막(h_max) 예측 log return (= raw_preds[-1])
#   Δy(t+h)  = raw_preds[h_idx]  (h시점 예측 log return)
#   chunk 간 기준가: h_max 시점 사다리꼴 예측값 사용

pred_dates   = future_dates

all_forecasts   = []
skipped_tickers = []

for ticker in tqdm(df_features['ticker'].unique(), desc="종목별 예측"):
    df_ticker = df_features[df_features['ticker'] == ticker].copy()
    df_ticker = df_ticker.sort_values('date').reset_index(drop=True)

    is_kospi_val = df_ticker['feature_is_kospi'].iloc[-1] \
        if 'feature_is_kospi' in df_ticker.columns else 0

    try:
        # ── Δy(t) 앵커 초기화 ─────────────────────────────────────
        # 첫 chunk의 Δy(t): 마지막 실측 log return
        # target_log_return_1d가 있으면 사용, 없으면 change_pct로 근사
        if 'target_log_return_1d' in df_ticker.columns:
            prev_delta_y = float(df_ticker['target_log_return_1d'].iloc[-1])
        elif 'change_pct' in df_ticker.columns:
            prev_delta_y = float(np.log1p(df_ticker['change_pct'].iloc[-1]))
        else:
            prev_delta_y = 0.0

        forecast_rows = []
        chunk_idx = 0

        # ── chunk 반복 ────────────────────────────────────────────
        while True:
            last_row = df_ticker.iloc[-1]

            X_pred    = df_ticker[feature_cols].iloc[[-1]]
            raw_preds = model.predict(X_pred)

            if isinstance(raw_preds, pd.DataFrame):
                raw_preds = raw_preds.values
            raw_preds = raw_preds.flatten()     # [pred_h1, ..., pred_h_max]

            # ── 역산 분기 ─────────────────────────────────────────
            if target_type == "log_return_1d":
                close_base     = last_row['close']
                log_close_base = np.log(max(close_base, 1e-9))
                delta_y_t      = prev_delta_y   # ✨ 사다리꼴 앵커

                for h_idx, h in enumerate(horizons):
                    pred_date = pred_dates[chunk_idx * len(horizons) + h_idx] \
                        if (chunk_idx * len(horizons) + h_idx) < len(pred_dates) else None
                    if pred_date is None:
                        break

                    pred_delta_h   = float(raw_preds[h_idx])    # Δy(t+h)
                    cum_log_return = float(np.sum(raw_preds[:h_idx + 1]))

                    # ✨ 사다리꼴: y(t+h) = y(t) + cumsum_h + (Δy(t) − Δy(t+h)) / 2
                    pred_log_close = log_close_base + cum_log_return + (delta_y_t - pred_delta_h) / 2
                    pred_close     = np.exp(pred_log_close)

                    row = {
                        'date'            : pred_date,
                        'ticker'          : ticker,
                        'horizon'         : h,
                        'chunk_idx'       : chunk_idx,
                        'pred_log_close'  : pred_log_close,
                        'pred_close'      : pred_close,
                        'pred_log_return' : pred_delta_h,   # 당일 등락률 로그값 (참고용)
                    }
                    forecast_rows.append(row)

                # ── chunk 간 기준가 갱신 ──────────────────────────
                # h_max 시점 사다리꼴 예측값을 다음 chunk의 기준가로 사용
                pred_delta_last    = float(raw_preds[-1])
                cum_log_return_all = float(np.sum(raw_preds))
                last_pred_log_close = (
                    log_close_base + cum_log_return_all + (delta_y_t - pred_delta_last) / 2
                )
                last_pred_close = np.exp(last_pred_log_close)

                # ✨ 다음 chunk의 Δy(t) = 이번 chunk h_max 예측 log return
                prev_delta_y = pred_delta_last

            else:  # log_close 모드 (기존)
                for h_idx, h in enumerate(horizons):
                    pred_date = pred_dates[chunk_idx * len(horizons) + h_idx] \
                        if (chunk_idx * len(horizons) + h_idx) < len(pred_dates) else None
                    if pred_date is None:
                        break

                    pred_log_close = float(raw_preds[h_idx])
                    pred_close     = np.exp(pred_log_close)

                    row = {
                        'date'           : pred_date,
                        'ticker'         : ticker,
                        'horizon'        : h,
                        'chunk_idx'      : chunk_idx,
                        'pred_log_close' : pred_log_close,
                        'pred_close'     : pred_close,
                    }
                    forecast_rows.append(row)

                last_pred_log_close = float(raw_preds[-1])
                last_pred_close     = np.exp(last_pred_log_close)

            # ── df_ticker에 예측 행 추가 (Recursive Extension) ───
            pred_date_h1 = pred_dates[chunk_idx * len(horizons)] \
                if chunk_idx * len(horizons) < len(pred_dates) else None
            if pred_date_h1 is None:
                break

            new_row = {
                'date'   : pred_date_h1,
                'ticker' : ticker,
                'close'  : last_pred_close,
                'volume' : df_ticker['volume'].iloc[-20:].mean(),
                **get_macro_row(pred_date_h1),
                'feature_is_kospi'  : is_kospi_val,
                'feature_is_monday' : int(pred_date_h1.weekday() == 0),
                'feature_is_friday' : int(pred_date_h1.weekday() == 4),
            }

            df_ticker = pd.concat(
                [df_ticker, pd.DataFrame([new_row])],
                ignore_index=True
            )
            df_ticker = calculate_features_for_ticker(df_ticker)

            chunk_idx += 1

            if chunk_idx * len(horizons) >= len(pred_dates):
                break

        all_forecasts.extend(forecast_rows)

    except Exception as e:
        skipped_tickers.append((ticker, type(e).__name__, str(e)))

In [ ]:
# ── 결과 요약 출력 ────────────────────────────────────────────
n_total   = df_features['ticker'].nunique()
n_skipped = len(skipped_tickers)
n_success = n_total - n_skipped

print(f"\n✅ Recursive Extension 완료")
print(f"   성공: {n_success:,}개 / 전체: {n_total:,}개")

if skipped_tickers:
    print(f"\n⚠️  제외된 종목: {n_skipped}개")
    print(f"   {'Ticker':<12} {'ErrorType':<25} {'Message'}")
    print(f"   {'-'*12} {'-'*25} {'-'*40}")
    for t, etype, msg in skipped_tickers:
        print(f"   {str(t):<12} {etype:<25} {msg[:60]}")
else:
    print("   제외된 종목 없음")

## 8️⃣ 예측 결과 저장

In [ ]:
df_forecasts = pd.DataFrame(all_forecasts)
df_forecasts = df_forecasts.sort_values(['ticker', 'date']).reset_index(drop=True)

forecast_parquet_path = paths.get_forecasts_parquet()
df_forecasts.to_parquet(forecast_parquet_path, index=False)

print(f"\n💾 예측 결과 저장 완료")
print(f"   - 파일: {forecast_parquet_path}")
print(f"   - 총 예측 행수: {len(df_forecasts):,}")

print("\n📊 예측 결과 샘플 (처음 10행):")
display(df_forecasts.head(10))

In [ ]:
SAVE_INDIVIDUAL_CSV = cfg['preprocessing'].get('save_csv', False)

if SAVE_INDIVIDUAL_CSV:
    csv_dir = paths.get_forecasts_csv_dir()
    csv_dir.mkdir(exist_ok=True)

    try:
        master_path    = paths.get_ticker_master()
        df_master      = pd.read_csv(master_path)
        ticker_name_map = dict(zip(df_master['ticker'].astype(str), df_master['name']))
    except Exception as e:
        print(f"   ⚠️  ticker_master 로드 실패: {e}")
        ticker_name_map = {}

    for ticker, group in tqdm(df_forecasts.groupby('ticker'), desc="CSV 저장"):
        name      = ticker_name_map.get(str(ticker), f"ticker_{ticker}")
        safe_name = str(name).replace('/', '_').replace('\\', '_')
        group.to_csv(csv_dir / f"{safe_name}_forecast.csv", index=False, encoding='utf-8-sig')

    print(f"   ✅ {df_forecasts['ticker'].nunique()}개 CSV 파일 저장 완료")
else:
    print("⏭️  개별 CSV 저장 건너뜀 (config.yaml에서 save_csv=True로 변경 가능)")

## 9️⃣ 예측 요약 통계

In [ ]:
print("\n" + "="*65)
print("📈 예측 결과 요약")
print("="*65)

print(f"\n[기본 정보]")
print(f"   - 총 종목 수:  {df_forecasts['ticker'].nunique():,}개")
print(f"   - 예측 기간:   {df_forecasts['date'].min().strftime('%Y-%m-%d')} ~ "
                         f"{df_forecasts['date'].max().strftime('%Y-%m-%d')}")
print(f"   - 총 예측 건수: {len(df_forecasts):,}건")

print(f"\n[예측 가격 통계]")
print(f"   - 평균 예측가: {df_forecasts['pred_close'].mean():,.0f}원")
print(f"   - 중앙값:      {df_forecasts['pred_close'].median():,.0f}원")
print(f"   - 최소값:      {df_forecasts['pred_close'].min():,.0f}원")
print(f"   - 최대값:      {df_forecasts['pred_close'].max():,.0f}원")

print(f"\n[Chunk별 예측 건수]")
chunk_counts = df_forecasts.groupby('chunk_idx').size()
for chunk_idx, count in chunk_counts.items():
    print(f"   - Chunk {chunk_idx}: {count:,}건")

print("\n" + "="*65)
print("✅ [Step 4] 미래 주가 예측 완료")
print("="*65)
print(f"\n💡 다음 단계: 05단계에서 Universe 필터링 및 투자 종목 선별")

## 🏁 완료 및 다음 단계

### ✅ 생성된 산출물
- `data/04_forecasts/{date}/{model_name}/future_forecasts.parquet`

### 🔁 실행 순서 (전체)
```
98_save_macro_data.ipynb
  → 97_forecast_macro.ipynb
    → 01_collect_data.ipynb
      → 02_build_dataset.ipynb
        → 03_train_predict.ipynb
          → 04_forecast_future.ipynb  ← 현재
            → 05_universe_selection.ipynb
```

---
**Last Updated**: v1.2 — v3.6.0 피처 스키마 동기화 + 매크로/정적 피처 미래값 반영  
**Pipeline Step**: 04 (Future Forecasting)  
**Method**: Recursive Extension (CHUNK_SIZE = max(horizons))